In [ ]:
import os
import glob
import re
import pandas as pd
import originpro as op
from collections import defaultdict

# =======================================================
# 🧪 1. 실험 조건 설정 (단위: 전류=mA, 면적=cm2 고정)
# =======================================================
target_folder = r"C:\Users\이승윤\Desktop\LSV"

REFERENCE_ELECTRODE = "Hg/HgO"  
E_0_REF = 0.197                  
PH_VALUE = 14                   
ELECTRODE_AREA = 0.5           # 전극 면적 (무조건 cm^2 단위로 입력)

# Rs 값 설정 (Ohm 단위)
RS_VALUES = {
    "Pt": 1.5,
    "Pd": 1.0
}
DEFAULT_RS = 0.0  

save_path = os.path.join(target_folder, "LSV_Compensated_mA_fixed.opju")
RHE_SHIFT = E_0_REF + (0.0592 * PH_VALUE)
# =======================================================

def process_mpt_fixed_units():
    mpt_files = glob.glob(os.path.join(target_folder, "*.mpt"))
    if not mpt_files:
        print("mpt 파일을 찾을 수 없습니다.")
        return

    file_groups = defaultdict(list)
    for file in mpt_files:
        fname = os.path.basename(file).replace(".mpt", "")
        match = re.search(r"^(.*)_(\d+)$", fname)
        if match:
            base_name = match.group(1)
            scan_num = int(match.group(2))
        else:
            base_name = fname
            scan_num = 0
        file_groups[base_name].append((scan_num, file, fname))

    for base in file_groups:
        file_groups[base].sort(key=lambda x: x[0])

    op.set_show(False)
    op.new()
    if op.find_book():
        op.find_book().destroy()

    for base_name, files_info in file_groups.items():
        print(f"➡ 그룹 처리 중: {base_name} (총 {len(files_info)}개 스캔)")
        
        dfs_per_scan = [] 
        
        for scan_num, file, original_fname in files_info:
            try:
                df_raw = pd.read_csv(file, sep=r'\s+', skiprows=62, header=None, encoding='latin1')
            except Exception as e:
                print(f"  [오류] {original_fname} 읽기 실패: {e}")
                continue

            pot_vs_ref = pd.to_numeric(df_raw.iloc[:, 6], errors='coerce')
            
            # [수정됨] 전류는 무조건 mA 스케일로 간주
            current_mA = pd.to_numeric(df_raw.iloc[:, 7], errors='coerce')

            valid_idx = pot_vs_ref.notna() & current_mA.notna()
            pot_vs_ref = pot_vs_ref[valid_idx]
            current_mA = current_mA[valid_idx]

            rs_value = RS_VALUES.get(original_fname, RS_VALUES.get(base_name, DEFAULT_RS))

            # --- 1. 원본 데이터 세팅 (4열) ---
            pot_vs_rhe = pot_vs_ref + RHE_SHIFT
            
            # [수정됨] 전류밀도 계산 (mA / cm2)
            current_density_mA = current_mA / ELECTRODE_AREA
            
            df_origin = pd.DataFrame({'A': pot_vs_ref, 'B': pot_vs_rhe, 'C': current_mA, 'D': current_density_mA})
            
            # [수정됨] Origin Unit을 mA와 mA/cm2로 고정
            df_origin.columns = pd.MultiIndex.from_tuples([
                (f'Potential vs. {REFERENCE_ELECTRODE}', 'V', original_fname),
                ('Potential vs. RHE', 'V', original_fname),
                ('Current', 'mA', original_fname),           
                ('Current Density', 'mA/cm2', original_fname) 
            ])
            df_origin.reset_index(drop=True, inplace=True)

            # --- 2. Rs 데이터 세팅 (1열) ---
            df_rs = pd.DataFrame({'Rs': [rs_value]})
            df_rs.columns = pd.MultiIndex.from_tuples([
                ('Series Resistance (Rs)', 'Ohm', f'{original_fname} Rs')
            ])

            # --- 3. 보상된 데이터 세팅 (2열) ---
            # [수정됨] V = A * Ohm 공식에 맞추기 위해 mA를 A로 1000 나누어 계산
            current_in_A = current_mA * 0.001
            pot_comp_ref = pot_vs_ref - (current_in_A * rs_value)
            pot_comp_rhe = pot_comp_ref + RHE_SHIFT

            df_comp = pd.DataFrame({'Comp_RHE': pot_comp_rhe, 'Comp_CD': current_density_mA})
            df_comp.columns = pd.MultiIndex.from_tuples([
                ('Potential vs. RHE (iR-free)', 'V', f'{original_fname} (Comp)'),
                ('Current Density', 'mA/cm2', f'{original_fname} (Comp)')
            ])
            df_comp.reset_index(drop=True, inplace=True)

            # --- 4. 7열 세트 결합 ---
            df_scan_combined = pd.concat([df_origin, df_rs, df_comp], axis=1)
            dfs_per_scan.append(df_scan_combined)

        if dfs_per_scan:
            df_final_group = pd.concat(dfs_per_scan, axis=1)
            wks = op.new_sheet('w', base_name)
            wks.from_df(df_final_group)

            axis_spec = ''.join(
                'X' if str(col_tuple[0]).startswith('Potential') else
                'Y' if str(col_tuple[0]).startswith('Current') else
                'N'
                for col_tuple in df_final_group.columns
            )
            wks.cols_axis(axis_spec, c1=0, c2=wks.cols - 1, repeat=False)
            print(f"  Origin X/Y 지정 패턴: {axis_spec}")

    op.save(save_path)
    print(f"\n✅ 완료! 파일 저장 위치:\n{save_path}")
    op.exit()

process_mpt_fixed_units()

➡ 그룹 처리 중: Pd (총 2개 스캔)
➡ 그룹 처리 중: Pt (총 2개 스캔)

✅ 완료! 파일 저장 위치:
C:\Users\이승윤\Desktop\LSV\LSV_Compensated_mA_fixed.opju


In [ ]:
import os
import glob
import re
import pandas as pd
import originpro as op
from collections import defaultdict

# =======================================================
# 🧪 1. 실험 조건 설정 (단위: 전류=mA, 면적=cm2 고정)
# =======================================================
target_folder = r"C:\Users\이승윤\Desktop\LSV"

REFERENCE_ELECTRODE = "Hg/HgO"  
E_0_REF = 0.197                  
PH_VALUE = 14                   
ELECTRODE_AREA = 0.5           # 전극 면적 (무조건 cm^2 단위로 입력)

# Rs 값 설정 (Ohm 단위)
RS_VALUES = {
    "Pt": 1.5,
    "Pd": 1.0
}
DEFAULT_RS = 0.0  

save_path = os.path.join(target_folder, "LSV_Compensated_Final2.opju")
RHE_SHIFT = E_0_REF + (0.059 * PH_VALUE)
# =======================================================

def process_mpt_final():
    mpt_files = glob.glob(os.path.join(target_folder, "*.mpt"))
    if not mpt_files:
        print("mpt 파일을 찾을 수 없습니다.")
        return

    file_groups = defaultdict(list)
    for file in mpt_files:
        fname = os.path.basename(file).replace(".mpt", "")
        match = re.search(r"^(.*)_(\d+)$", fname)
        if match:
            base_name = match.group(1)
            scan_num = int(match.group(2))
        else:
            base_name = fname
            scan_num = 0
        file_groups[base_name].append((scan_num, file, fname))

    for base in file_groups:
        file_groups[base].sort(key=lambda x: x[0])

    op.set_show(False)
    op.new()
    if op.find_book():
        op.find_book().destroy()

    for base_name, files_info in file_groups.items():
        print(f"➡ 그룹 처리 중: {base_name}")
        
        dfs_per_scan = [] 
        
        for scan_num, file, original_fname in files_info:
            try:
                df_raw = pd.read_csv(file, sep=r'\s+', skiprows=62, header=None, encoding='latin1')
            except Exception as e:
                print(f"  [오류] {original_fname} 읽기 실패: {e}")
                continue

            pot_vs_ref = pd.to_numeric(df_raw.iloc[:, 6], errors='coerce')
            current_mA = pd.to_numeric(df_raw.iloc[:, 7], errors='coerce')

            valid_idx = pot_vs_ref.notna() & current_mA.notna()
            pot_vs_ref = pot_vs_ref[valid_idx]
            current_mA = current_mA[valid_idx]

            rs_value = RS_VALUES.get(original_fname, RS_VALUES.get(base_name, DEFAULT_RS))

            # --- 데이터 세팅 ---
            pot_vs_rhe = pot_vs_ref + RHE_SHIFT
            current_density_mA = current_mA / ELECTRODE_AREA
            
            df_origin = pd.DataFrame({'A': pot_vs_ref, 'B': pot_vs_rhe, 'C': current_mA, 'D': current_density_mA})
            df_origin.columns = pd.MultiIndex.from_tuples([
                (f'Potential vs. {REFERENCE_ELECTRODE}', 'V', original_fname),
                ('Potential vs. RHE', 'V', original_fname),
                ('Current', 'mA', original_fname),           
                ('Current Density', 'mA/cm2', original_fname) 
            ])
            df_origin.reset_index(drop=True, inplace=True)

            df_rs = pd.DataFrame({'Rs': [rs_value]})
            df_rs.columns = pd.MultiIndex.from_tuples([
                ('Series Resistance (Rs)', 'Ohm', f'{original_fname} Rs')
            ])

            current_in_A = current_mA * 0.001
            pot_comp_ref = pot_vs_ref - (current_in_A * rs_value)
            pot_comp_rhe = pot_comp_ref + RHE_SHIFT

            df_comp = pd.DataFrame({'Comp_RHE': pot_comp_rhe, 'Comp_CD': current_density_mA})
            df_comp.columns = pd.MultiIndex.from_tuples([
                ('Potential vs. RHE (iR-free)', 'V', f'{original_fname} (Comp)'),
                ('Current Density', 'mA/cm2', f'{original_fname} (Comp)')
            ])
            df_comp.reset_index(drop=True, inplace=True)

            df_scan_combined = pd.concat([df_origin, df_rs, df_comp], axis=1)
            dfs_per_scan.append(df_scan_combined)

        if dfs_per_scan:
            df_final_group = pd.concat(dfs_per_scan, axis=1)
            
            # Origin 시트 생성
            wks = op.new_sheet('w', base_name)
            
            # 데이터를 Origin에 넣고 열 라벨을 설정
            wks.from_df(df_final_group)
            for i, col_tuple in enumerate(df_final_group.columns):
                wks.set_label(i, str(col_tuple[0]), 'L')
                wks.set_label(i, str(col_tuple[1]), 'U')
                wks.set_label(i, str(col_tuple[2]), 'C')

            axis_spec = ''.join(
                'X' if str(col_tuple[0]).startswith('Potential') else
                'Y' if str(col_tuple[0]).startswith('Current') else
                'N'
                for col_tuple in df_final_group.columns
            )
            wks.cols_axis(axis_spec, c1=0, c2=wks.cols - 1, repeat=False)
            print(f"  Origin X/Y 지정 패턴: {axis_spec}")

    op.save(save_path)
    print(f"\n✅ 완료! 파일 저장 위치:\n{save_path}")
    op.exit()

process_mpt_final()

➡ 그룹 처리 중: Pd
➡ 그룹 처리 중: Pt

✅ 완료! 파일 저장 위치:
C:\Users\이승윤\Desktop\LSV\LSV_Compensated_Final2.opju
